In [134]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from pathlib import Path
import json
from typing import Any, Dict, List, Tuple, Union, Optional
import re

In [135]:
nan_list = [None, [], {}, 'NaN', 'Null','NULL','None', 'none', 'nulo', 'NA','<NA>','NaT','?','-', '.','', ' ', '   ', 'unknown', 'Unknown','[unknown]']

In [136]:
def data_exploratory_analysis(data: pd.DataFrame) -> None:
    
    """
    Perform Data Brief inspection.
    """
        
    print(f"===== Data Inspection =====")

    print(f"\nShape:\n{data.shape}")

    print(f"\nData Types:\n{data.dtypes}")
    
    print ('Number of Null Rows:', data.isna().sum().sum())

    print(f"\nMissing Values:\n{data.isnull().sum()}")

    print(f"\nUnique Values:\n{data.nunique()}")

    repeated_rows = data[data.duplicated()]
    
    print(f"\nNumber of Repeated Rows: {len(repeated_rows)}")

    print("\n" + "="*40 + "\n")



def check_missing_values(data: pd.DataFrame, nan_list: List[str]) -> None:
    
    """
    This function looks for all types of missing values in DataFrame columns,
    including a predefined list of values that are considered as "missing".
    """
    
    for c in data.columns:
        mask = data[c].isin(nan_list[2:])
        print(f"Column '{c}': {data[c].isnull().sum()} missing, {mask.sum()} from nan_list")



def print_null_percentages_and_return_columns(data: pd.DataFrame) -> List[str]:
    
    """
    Prints the percentage of null values for each column in the given DataFrame where the percentage is above 0%.
    Returns a list of column names that have null values.
    """
    
    total_rows = len(data)
    cols_with_nulls = [] 
    
    for column in data.columns:
        null_count = data[column].isnull().sum()
        
        if null_count > 0:  
            null_percentage = (null_count / total_rows) * 100
            print(f"{column}: {null_percentage:.2f}% null values")
            cols_with_nulls.append(column)  

    return cols_with_nulls


# Jogos

In [137]:
diretorio = Path("../api_outputs/jogos")

dataframes = []

for arquivo_json in diretorio.glob("202*/*.json"):
    with open(arquivo_json, "r", encoding="utf-8") as f:
        dados = json.load(f)

    df_arquivo = pd.DataFrame(dados)
    dataframes.append(df_arquivo)

df_jogos = pd.concat(dataframes, ignore_index=True)
print(df_jogos.shape)
df_jogos.head()

(1440, 5)


,jogo_id,edicao,rodada,equipe_mandante_id,equipe_visitante_id
0,900001,2022,1,111,114
1,900002,2022,1,105,109
2,900003,2022,1,108,120
3,900004,2022,1,107,124
4,900005,2022,1,102,103


# Atletas

In [138]:
diretorio = Path("../api_outputs/atletas")

dataframes = []

for arquivo_json in diretorio.glob("*.json"):
    with open(arquivo_json, "r", encoding="utf-8") as f:
        dados = json.load(f)

    df_arquivo = pd.DataFrame([dados])
    dataframes.append(df_arquivo)

df_atletas = pd.concat(dataframes, ignore_index=True)
print(df_atletas.shape)
df_atletas.head()

(2031, 4)


,atleta_id,apelido,posicao_id,clube_id
0,11201,Geraldo Bittencourt,3,109
1,10943,Zeca Henriques,1,107
2,11651,Zeca Carvalho,2,104
3,10410,Zeca Siqueira,5,122
4,10040,Rafael Peixoto,5,120


# Confrontos

In [139]:
diretorio = Path("../api_outputs/confrontos")

dataframes = []

for arquivo_json in diretorio.glob("202*/*.json"):
    nome = arquivo_json.stem
    padrao = r"confrontos_temporada_(\d+)_rodada_(\d+)"

    match = re.search(padrao, nome)
    if not match:
        print(f"[AVISO] Nome de arquivo fora do padrão esperado, ignorado: {arquivo_json.name}")
        continue

    temporada = match.group(1)
    rodada = match.group(2)

    with open(arquivo_json, "r", encoding="utf-8") as f:
        dados = json.load(f)

    df_arquivo = pd.DataFrame(dados)
    df_arquivo["temporada"] = temporada
    df_arquivo["rodada"] = rodada

    dataframes.append(df_arquivo)

df_confrontos = pd.concat(dataframes, ignore_index=True)
df_confrontos['temporada'] = df_confrontos['temporada'].astype(int)
df_confrontos['rodada'] = df_confrontos['rodada'].astype(int)
print(df_confrontos.shape)
df_confrontos.head()

(2784, 6)


,equipe_id,adversario_id,equipe_media_pontos_conquistados,adversario_media_pontos_cedidos,temporada,rodada
0,101,114,46.25,40.98,2022,11
1,102,122,46.05,52.91,2022,11
2,103,115,51.42,53.97,2022,11
3,105,126,50.68,55.02,2022,11
4,107,121,61.44,42.91,2022,11


# Equipes

In [140]:
diretorio = Path("../api_outputs/equipes")

dataframes = []

for arquivo_json in diretorio.glob("*.json"):
    with open(arquivo_json, "r", encoding="utf-8") as f:
        dados = json.load(f)

    df_arquivo = pd.DataFrame(dados)
    dataframes.append(df_arquivo)

df_equipes = pd.concat(dataframes, ignore_index=True)
print(df_equipes.shape)
df_equipes.head()

(28, 3)


,equipe_id,nome,sigla
0,101,Clube 101,C101
1,102,Clube 102,C102
2,103,Clube 103,C103
3,104,Clube 104,C104
4,105,Clube 105,C105


# Escalações

In [141]:
diretorio = Path("../api_outputs/escalacoes")

registros = []
arquivos_vazios_ou_invalidos = []

for arquivo_json in diretorio.glob("*.json"):
    jogo_id = arquivo_json.stem.replace('jogos_', '').replace('.json', '')

    with open(arquivo_json, "r", encoding="utf-8") as f:
        dados = json.load(f)

    if not isinstance(dados, dict):
        arquivos_vazios_ou_invalidos.append(arquivo_json.name)
        continue

    for equipe_id, equipe in dados.items():
        if not isinstance(equipe, dict):
            continue

        titulares = equipe.get("titulares") or []
        reservas = equipe.get("reservas") or []

        for jogador in titulares:
            if not isinstance(jogador, dict) or "atleta_id" not in jogador:
                continue
            registros.append({
                "jogo_id": jogo_id,
                "equipe_id": equipe_id,
                "atleta_id": jogador["atleta_id"],
                "titular": True,
                "momento_substituido": (jogador.get("substituido") or {}).get("momento"),
                "momento_entrou": None,
            })

        for jogador in reservas:
            if not isinstance(jogador, dict) or "atleta_id" not in jogador:
                continue
            registros.append({
                "jogo_id": jogo_id,
                "equipe_id": equipe_id,
                "atleta_id": jogador["atleta_id"],
                "titular": False,
                "momento_substituido": None,
                "momento_entrou": (jogador.get("entrou") or {}).get("momento"),
            })

df_escalacoes = pd.DataFrame(registros)

print(f"Total de registros: {df_escalacoes.shape[0]}")
print(f"Arquivos sem escalação (raiz não-dict): {len(arquivos_vazios_ou_invalidos)}")

if arquivos_vazios_ou_invalidos:
    print(arquivos_vazios_ou_invalidos[:10], "..." if len(arquivos_vazios_ou_invalidos) > 10 else "")

print(df_escalacoes.shape)
ordem = ['atleta_id',  'equipe_id', 'jogo_id', 'titular', 'momento_substituido', 'momento_entrou']
df_escalacoes = df_escalacoes[ordem]
df_escalacoes['equipe_id'] = df_escalacoes['equipe_id'].astype(int)
df_escalacoes['jogo_id'] = df_escalacoes['jogo_id'].astype(int)
df_escalacoes.head()

Total de registros: 101650
Arquivos sem escalação (raiz não-dict): 63
['jogos_901483.json', 'jogos_901270.json', 'jogos_901118.json', 'jogos_900924.json', 'jogos_901266.json', 'jogos_901267.json', 'jogos_900821.json', 'jogos_900118.json', 'jogos_901115.json', 'jogos_900801.json'] ...
(101650, 6)


,atleta_id,equipe_id,jogo_id,titular,momento_substituido,momento_entrou
0,10114,108,901354,True,49min,NaN
1,10125,108,901354,True,NaN,NaN
2,10309,108,901354,True,70min,NaN
3,10333,108,901354,True,NaN,NaN
4,10562,108,901354,True,NaN,NaN


# Base gato-mestre

In [142]:
base_gm = pd.read_csv('../../material_apoio/base_case_gm.csv')
base_gm.head()

,atleta_id,apelido,ano,rodada_id,clube_id,posicao_id,status_pre,status_inicial,preco_num,variacao_num,...,FS,PS,GS,GC,CA,CV,FC,I,PP,PC
0,11633,Kaique Oliveira,2024,20,127,4,Nulo,reserva,1.98,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,11798,Geraldo Antunes,2025,13,110,6,PROVÁVEL,0,9.39,0.36,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,10470,Pedro Peixoto,2024,14,127,3,Nulo,titular,2.8,-1.02,...,0.0,0.0,0.0,0.0,1.0,0.0,2.0,0.0,0.0,0.0
3,10661,Adriel Barbosa,2025,36,115,4,Nulo,reserva,1.87,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,10218,Norberto Guimarães,2022,30,118,5,Nulo,reserva,5.55,1.34,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


# Join entre as bases

In [143]:
join1 = base_gm.merge(
    df_jogos,
    left_on="match_id",
    right_on="jogo_id",
    how="left",
    suffixes=("", "_jogos"),
)

join2 = join1.merge(
    df_confrontos,
    left_on=["clube_id", "opponent", "ano", "rodada_id"],
    right_on=["equipe_id", "adversario_id", "temporada", "rodada"],
    how="left",
    suffixes=("", "_confrontos"),
)

join3 = join2.merge(
    df_escalacoes,
    left_on=["atleta_id", "match_id"],
    right_on=["atleta_id", "jogo_id"],
    how="left",
    suffixes=("", "_escalacoes"),
)

dados = join3

dados['preco_num'] = dados['preco_num'].str.replace(',', '.').astype(float)
dados['equipe_media_pontos_conquistados_ausencia'] = dados.shape[0] * [0]
dados['adversario_media_pontos_cedidos_ausencia'] = dados.shape[0] * [0]
dados['minutos_jogados_ausencia'] = dados.shape[0] * [0]
dados['momento_entrou_ausencia'] = dados.shape[0] * [0]
dados['momento_substituido_ausencia'] = dados.shape[0] * [0]

linhas_duplicadas = dados.duplicated(subset=["atleta_id", "match_id"]).sum()
linhas_esperadas = len(base_gm)
linhas_obtidas = len(dados)

print(f"Linhas em base_gm (entrada): {linhas_esperadas}")
print(f"Linhas em dados (saída): {linhas_obtidas}")
print(f"Linhas duplicadas em (atleta_id, match_id): {linhas_duplicadas}")

print("Colunas finais:", list(dados.columns))

Linhas em base_gm (entrada): 117469
Linhas em dados (saída): 117469
Linhas duplicadas em (atleta_id, match_id): 1856
Colunas finais: ['atleta_id', 'apelido', 'ano', 'rodada_id', 'clube_id', 'posicao_id', 'status_pre', 'status_inicial', 'preco_num', 'variacao_num', 'media_num', 'jogos_num', 'pontos_num', 'minutos_jogados', 'entrou_em_campo', 'home_dummy', 'opponent', 'match_id', 'G', 'A', 'SG', 'FF', 'FT', 'FD', 'DD', 'DP', 'DE', 'DS', 'FS', 'PS', 'GS', 'GC', 'CA', 'CV', 'FC', 'I', 'PP', 'PC', 'jogo_id', 'edicao', 'rodada', 'equipe_mandante_id', 'equipe_visitante_id', 'equipe_id', 'adversario_id', 'equipe_media_pontos_conquistados', 'adversario_media_pontos_cedidos', 'temporada', 'rodada_confrontos', 'equipe_id_escalacoes', 'jogo_id_escalacoes', 'titular', 'momento_substituido', 'momento_entrou', 'equipe_media_pontos_conquistados_ausencia', 'adversario_media_pontos_cedidos_ausencia', 'minutos_jogados_ausencia', 'momento_entrou_ausencia', 'momento_substituido_ausencia']


In [144]:
dados.head()

,atleta_id,apelido,ano,rodada_id,clube_id,posicao_id,status_pre,status_inicial,preco_num,variacao_num,...,equipe_id_escalacoes,jogo_id_escalacoes,titular,momento_substituido,momento_entrou,equipe_media_pontos_conquistados_ausencia,adversario_media_pontos_cedidos_ausencia,minutos_jogados_ausencia,momento_entrou_ausencia,momento_substituido_ausencia
0,11633,Kaique Oliveira,2024,20,127,4,Nulo,reserva,1.98,0.00,...,127.0,900849.0,False,NaN,NaN,0,0,0,0,0
1,11798,Geraldo Antunes,2025,13,110,6,PROVÁVEL,0,9.39,0.36,...,110.0,901223.0,False,NaN,0min,0,0,0,0,0
2,10470,Pedro Peixoto,2024,14,127,3,Nulo,titular,2.80,-1.02,...,127.0,901049.0,True,89min,NaN,0,0,0,0,0
3,10661,Adriel Barbosa,2025,36,115,4,Nulo,reserva,1.87,0.00,...,115.0,901290.0,False,NaN,NaN,0,0,0,0,0
4,10218,Norberto Guimarães,2022,30,118,5,Nulo,reserva,5.55,1.34,...,118.0,900097.0,False,NaN,51min,0,0,0,0,0


In [145]:
data_exploratory_analysis(dados)

===== Data Inspection =====

Shape:
(117469, 59)

Data Types:
atleta_id                                      int64
apelido                                          str
ano                                            int64
rodada_id                                      int64
clube_id                                       int64
posicao_id                                     int64
status_pre                                       str
status_inicial                                   str
preco_num                                    float64
variacao_num                                 float64
media_num                                    float64
jogos_num                                      int64
pontos_num                                   float64
minutos_jogados                              float64
entrou_em_campo                                 bool
home_dummy                                   float64
opponent                                     float64
match_id                             

In [146]:
colunas = [
    'adversario_id',
    'edicao',
    'equipe_id',
    'equipe_id_escalacoes',
    'home_dummy',
    'jogo_id_escalacoes',
    'opponent',
    'rodada',
    'rodada_confrontos',
    'temporada',
    'titular',
    'jogo_id'
]

dados = dados.drop(columns=colunas)

In [147]:
data_exploratory_analysis(dados)

===== Data Inspection =====

Shape:
(117469, 47)

Data Types:
atleta_id                                      int64
apelido                                          str
ano                                            int64
rodada_id                                      int64
clube_id                                       int64
posicao_id                                     int64
status_pre                                       str
status_inicial                                   str
preco_num                                    float64
variacao_num                                 float64
media_num                                    float64
jogos_num                                      int64
pontos_num                                   float64
minutos_jogados                              float64
entrou_em_campo                                 bool
match_id                                       int64
G                                            float64
A                                    

In [148]:
dados = dados.dropna(subset=['equipe_mandante_id', 'equipe_visitante_id'])

In [149]:
data_exploratory_analysis(dados)

===== Data Inspection =====

Shape:
(111720, 47)

Data Types:
atleta_id                                      int64
apelido                                          str
ano                                            int64
rodada_id                                      int64
clube_id                                       int64
posicao_id                                     int64
status_pre                                       str
status_inicial                                   str
preco_num                                    float64
variacao_num                                 float64
media_num                                    float64
jogos_num                                      int64
pontos_num                                   float64
minutos_jogados                              float64
entrou_em_campo                                 bool
match_id                                       int64
G                                            float64
A                                    

In [150]:
# preco_num
dados["preco_num"] = dados["preco_num"].fillna(
    dados.groupby("atleta_id")["preco_num"].transform("mean")
)

# minutos_jogados
dados.loc[
    (dados['minutos_jogados'].isna()) & (dados['entrou_em_campo'] == True),
    'minutos_jogados'
] = 90.0

dados.loc[
    (dados['minutos_jogados'].isna()) & (dados['entrou_em_campo'] != True),
    'minutos_jogados'
] = 0.0

dados.loc[
    (dados['status_inicial'] == '0'),
    'status_inicial'
] = 'reserva'

# momento_entrou
dados.loc[
    (dados['momento_entrou'].isna()) & (dados['entrou_em_campo'] == True),
    'momento_entrou'
] = "0min"

dados.loc[
    (dados['momento_entrou'].isna()) & (dados['entrou_em_campo'] != True),
    'momento_entrou'
] = "0min"

dados['momento_entrou'] = dados['momento_entrou'].str.replace('min', '').astype(int)

# momento_substituido
dados.loc[
    (dados['momento_substituido'].isna()) & (dados['entrou_em_campo'] == True),
    ['momento_substituido', 'momento_substituido_ausencia']
] = ["0min", 0]

dados.loc[
    (dados['momento_substituido'].isna()) & (dados['entrou_em_campo'] != True),
    ['momento_substituido', 'momento_substituido_ausencia']
] = ["0min", 1]

dados['momento_substituido'] = dados['momento_substituido'].str.replace('min', '').astype(int)

# equipe_media_pontos_conquistados
dados.loc[
    (dados['rodada_id'] == 1) & (dados['equipe_media_pontos_conquistados'].isna()),
    ['equipe_media_pontos_conquistados', 'equipe_media_pontos_conquistados_ausencia']
] = [0.0, 0]

dados.loc[
    (dados['rodada_id'] != 1) & (dados['equipe_media_pontos_conquistados'].isna()),
    ['equipe_media_pontos_conquistados', 'equipe_media_pontos_conquistados_ausencia']
] = [0.0, 1]

# adversario_media_pontos_cedidos
dados.loc[
    (dados['rodada_id'] == 1) & (dados['adversario_media_pontos_cedidos'].isna()),
    ['adversario_media_pontos_cedidos', 'adversario_media_pontos_cedidos_ausencia']
] = [0.0, 0]

dados.loc[
    (dados['rodada_id'] != 1) & (dados['adversario_media_pontos_cedidos'].isna()),
    ['adversario_media_pontos_cedidos', 'adversario_media_pontos_cedidos_ausencia']
] = [0.0, 1]

# status_pre
dados = dados.drop(columns=['status_pre', 'apelido', 'DD'])

# dummies da coluna status_inicial
dados = pd.get_dummies(dados, columns=['status_inicial'])

In [162]:
data_exploratory_analysis(dados)

===== Data Inspection =====

Shape:
(111720, 47)

Data Types:
atleta_id                                      int64
ano                                            int64
rodada_id                                      int64
clube_id                                       int64
posicao_id                                     int64
preco_num                                    float64
variacao_num                                 float64
media_num                                    float64
jogos_num                                      int64
pontos_num                                   float64
minutos_jogados                              float64
entrou_em_campo                                 bool
match_id                                       int64
G                                            float64
A                                            float64
SG                                           float64
FF                                           float64
FT                                   

In [155]:
# dados.drop(columns=[
#     'atleta_id', 
#     'ano',
#     'clube_id',
#     'rodada_id', 
#     'posicao_id', 
#     'match_id', 
#     'equipe_mandante_id',
#     'equipe_visitante_id',
#     'equipe_media_pontos_conquistados_ausencia',
#     'adversario_media_pontos_cedidos_ausencia',
#     'minutos_jogados_ausencia',
#     'momento_entrou_ausencia',
#     'momento_substituido_ausencia'
# ]).describe()

dados[['A',
 'CA',
 'CV',
 'DD',
 'DE',
 'DP',
 'DS',
 'FC',
 'FD',
 'FF',
 'FS',
 'FT',
 'G',
 'GC',
 'GS',
 'I',
 'PC',
 'PP',
 'PS',
 'SG']].describe()

,A,CA,CV,DD,DE,DP,DS,FC,FD,FF,FS,FT,G,GC,GS,I,PC,PP,PS,SG
count,111720.000000,111720.000000,111720.000000,111720.0,111720.000000,111720.000000,111720.000000,111720.000000,111720.000000,111720.000000,111720.000000,111720.000000,111720.000000,111720.000000,111720.000000,111720.000000,111720.000000,111720.000000,111720.000000,111720.000000
mean,0.020158,0.061654,0.003589,0.0,0.081006,0.000501,0.476298,0.406749,0.080541,0.131883,0.318654,0.008378,0.029377,0.000662,0.030174,0.039223,0.003634,0.000716,0.002685,0.042392
std,0.147743,0.240527,0.059804,0.0,0.598267,0.022383,0.990070,0.826798,0.324600,0.432429,0.799002,0.093284,0.181582,0.025728,0.256204,0.230897,0.060619,0.026750,0.052095,0.201482
min,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,3.000000,1.000000,1.000000,0.0,13.000000,1.000000,10.000000,9.000000,5.000000,7.000000,9.000000,3.000000,3.000000,1.000000,8.000000,6.000000,2.000000,1.000000,2.000000,1.000000


In [160]:
dados[(dados['status_inicial_titular'] == True)].drop(columns=[
    'atleta_id', 
    'ano',
    'clube_id',
    'rodada_id', 
    'posicao_id', 
    'match_id', 
    'equipe_mandante_id',
    'equipe_visitante_id',
    'equipe_media_pontos_conquistados_ausencia',
    'adversario_media_pontos_cedidos_ausencia',
    'minutos_jogados_ausencia',
    'momento_entrou_ausencia',
    'momento_substituido_ausencia'
])[['A',
 'CA',
 'CV',
 'DD',
 'DE',
 'DP',
 'DS',
 'FC',
 'FD',
 'FF',
 'FS',
 'FT',
 'G',
 'GC',
 'GS',
 'I',
 'PC',
 'PP',
 'PS',
 'SG']].describe()

,preco_num,variacao_num,media_num,jogos_num,pontos_num,minutos_jogados,G,A,SG,FF,...,CA,CV,FC,I,PP,PC,equipe_media_pontos_conquistados,adversario_media_pontos_cedidos,momento_substituido,momento_entrou
count,29176.000000,29176.000000,29176.000000,29176.000000,29176.000000,29176.000000,29176.000000,29176.000000,29176.000000,29176.000000,...,29176.000000,29176.000000,29176.000000,29176.000000,29176.000000,29176.000000,29176.000000,29176.000000,29176.000000,29176.0
mean,6.775645,0.065658,3.675371,11.982794,3.795590,98.483000,0.084899,0.059158,0.127434,0.381821,...,0.176069,0.009631,1.001268,0.114683,0.002296,0.011345,42.963191,43.067300,26.180457,0.0
std,3.374275,0.973841,1.981934,8.010459,3.998081,105.269481,0.303914,0.249620,0.333464,0.682114,...,0.380886,0.097667,1.138332,0.391962,0.047867,0.106875,23.117948,22.864547,34.759722,0.0
min,-18.990000,-7.190000,-4.300000,0.000000,-8.290000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
25%,4.620000,-0.480000,2.380000,5.000000,0.900000,79.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,37.440000,39.020000,0.000000,0.0
50%,6.320000,0.000000,3.470000,11.000000,2.800000,95.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,49.790000,49.700000,0.000000,0.0
75%,8.530000,0.520000,4.720000,18.000000,5.900000,100.000000,0.000000,0.000000,0.000000,1.000000,...,0.000000,0.000000,2.000000,0.000000,0.000000,0.000000,57.590000,57.500000,66.000000,0.0
max,29.110000,13.830000,27.100000,35.000000,33.500000,1440.000000,3.000000,3.000000,1.000000,7.000000,...,1.000000,1.000000,9.000000,6.000000,1.000000,2.000000,118.710000,118.710000,89.000000,0.0


In [163]:
def winsorizar_coluna(coluna, inferior=0.01, superior=0.99):
    limite_inferior = coluna.quantile(inferior)
    limite_superior = coluna.quantile(superior)

    return coluna.clip(
        lower=limite_inferior,
        upper=limite_superior
    )


dados['minutos_jogados'] = winsorizar_coluna(dados['minutos_jogados'], 0.02, 0.98)

In [165]:
dados.drop(columns=[
    'atleta_id', 
    'ano',
    'clube_id',
    'rodada_id', 
    'posicao_id', 
    'match_id', 
    'equipe_mandante_id',
    'equipe_visitante_id',
    'equipe_media_pontos_conquistados_ausencia',
    'adversario_media_pontos_cedidos_ausencia',
    'minutos_jogados_ausencia',
    'momento_entrou_ausencia',
    'momento_substituido_ausencia'
]).describe()

,preco_num,variacao_num,media_num,jogos_num,pontos_num,minutos_jogados,G,A,SG,FF,...,CA,CV,FC,I,PP,PC,equipe_media_pontos_conquistados,adversario_media_pontos_cedidos,momento_substituido,momento_entrou
count,111720.000000,111720.000000,111720.000000,111720.000000,111720.000000,111720.000000,111720.000000,111720.000000,111720.000000,111720.000000,...,111720.000000,111720.000000,111720.000000,111720.000000,111720.000000,111720.000000,111720.000000,111720.000000,111720.000000,111720.000000
mean,4.826690,0.000214,2.215247,7.210482,1.370776,30.324114,0.029377,0.020158,0.042392,0.131883,...,0.061654,0.003589,0.406749,0.039223,0.000716,0.003634,43.075736,43.261165,7.199857,7.064017
std,3.337335,0.584440,2.201684,7.697722,2.916432,41.214872,0.181582,0.147743,0.201482,0.432429,...,0.240527,0.059804,0.826798,0.230897,0.026750,0.060619,22.598748,22.407370,21.650822,20.340849
min,-19.000000,-7.190000,-5.400000,0.000000,-8.870000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2.340000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,38.000000,39.640000,0.000000,0.000000
50%,4.280000,0.000000,1.940000,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,49.640000,49.700000,0.000000,0.000000
75%,6.650000,0.000000,3.640000,12.000000,1.500000,75.250000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,57.230000,57.120000,0.000000,0.000000
max,29.110000,13.830000,27.100000,36.000000,33.500000,105.000000,3.000000,3.000000,1.000000,7.000000,...,1.000000,1.000000,9.000000,6.000000,1.000000,2.000000,118.710000,118.710000,89.000000,89.000000


In [166]:
set(dados.columns)

{'A',
 'CA',
 'CV',
 'DD',
 'DE',
 'DP',
 'DS',
 'FC',
 'FD',
 'FF',
 'FS',
 'FT',
 'G',
 'GC',
 'GS',
 'I',
 'PC',
 'PP',
 'PS',
 'SG',
 'adversario_media_pontos_cedidos',
 'adversario_media_pontos_cedidos_ausencia',
 'ano',
 'atleta_id',
 'clube_id',
 'entrou_em_campo',
 'equipe_mandante_id',
 'equipe_media_pontos_conquistados',
 'equipe_media_pontos_conquistados_ausencia',
 'equipe_visitante_id',
 'jogos_num',
 'match_id',
 'media_num',
 'minutos_jogados',
 'minutos_jogados_ausencia',
 'momento_entrou',
 'momento_entrou_ausencia',
 'momento_substituido',
 'momento_substituido_ausencia',
 'pontos_num',
 'posicao_id',
 'preco_num',
 'rodada_id',
 'status_inicial_nao_relacionado',
 'status_inicial_reserva',
 'status_inicial_titular',
 'variacao_num'}